# Лабораторна робота № 5
## Географічні дані OpenStreetMap: від тегів до маршруту

**Інтелектуальний аналіз та візуалізація даних** · 4 години · 4 бали

---

### Постановка

Ви — аналітик у туристичному застосунку. Завдання: для заданого міста
**відібрати десять найвартісніших історико-культурних об'єктів** і
**спланувати піший маршрут**, який починається на залізничному вокзалі,
обходить усі десять і повертається на вокзал.

Слово «найвартісніші» в даних не записане. Критерій ви побудуєте самі —
і захистите його.

Потім те саме для **тридцяти** об'єктів. Результат цієї, більшої задачі
стане базою порівняння в **лабораторній роботі № 8**, де ту саму задачу
розв'язуватимуть генетичним та мурашиним алгоритмами.

### Що здаєте

1. Цей ноутбук із виконаними комірками (усі виводи збережені).
2. Файл `lr5_результати.json` — результати для ЛР8.
3. Карту `lr5_маршрут.html`.
4. Звіт у кінці ноутбука: відповіді на сім питань.

### Що оцінюється

| критерій | 0 | 2 | 3 | 4 |
|---|---|---|---|---|
| Відбір об'єктів | теги взято навмання | фільтр працює | фільтр обґрунтовано | показано, що відкинуто і чому |
| Критерій рейтингу | немає | є формула | ваги пояснено | порівняно з альтернативним критерієм |
| Маршрут | не будується | будується | перевірено коректність | пояснено різницю методів числом |
| Карта | немає | точки на карті | маршрут по ребрах графа | видно різницю з прямими відрізками |
| Звіт | немає | формально | з числами | з висновком, якого не було в завданні |

---
## 0. Ваш варіант

Впишіть прізвище та номер у списку групи. Місто підтягнеться само.

In [46]:
STUDENT_NAME = 'Паничевська Ярина Ернестівна'
LIST_NUMBER  = 15

CITIES = [
    'Львів', 'Київ', 'Харків', 'Одеса', 'Дніпро',
    'Чернівці', 'Івано-Франківськ', 'Тернопіль', 'Ужгород', 'Луцьк',
    'Рівне', 'Хмельницький', 'Вінниця', 'Житомир', 'Чернігів',
    'Черкаси', 'Полтава', 'Суми', 'Кропивницький', 'Запоріжжя',
    'Миколаїв', 'Кривий Ріг', 'Кам\u2019янець-Подільський', 'Мукачево',
    'Коломия', 'Дрогобич', 'Біла Церква', 'Ніжин', 'Остріг', 'Умань',
]

assert STUDENT_NAME and LIST_NUMBER > 0, 'впишіть прізвище та номер у списку'
CITY = CITIES[(LIST_NUMBER - 1) % len(CITIES)] + ', Україна'
print('варіант %d: %s' % (LIST_NUMBER, CITY))

варіант 15: Чернігів, Україна


> Якщо викладач призначив вам інше місто — просто перепишіть `CITY` вручну
> і зазначте це у звіті.

In [47]:
import importlib.util, subprocess, sys
REQUIRED = ['osmnx', 'geopandas', 'shapely', 'pyproj', 'pyarrow',
            'folium', 'mapclassify', 'h3', 'networkx']
missing = [p for p in REQUIRED if importlib.util.find_spec(p) is None]
if missing:
    print('встановлюю:', ', '.join(missing), '...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('усі пакети на місці')

усі пакети на місці


In [48]:
import warnings; warnings.filterwarnings('ignore')
import json, time, os, collections
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
from xyzservices import providers as xyz

BASEMAP = xyz.Esri.WorldGrayCanvas
pd.set_option('display.width', 200)
print('geopandas', gpd.__version__)

geopandas 1.2.0


---
## 1. Два способи дістати дані

Робоча програма вимагає порівняти два способи доступу **за обсягом переданих
даних і за часом**:

* **живий запит** — `osmnx` питає Overpass API просто зараз;
* **знімок** — ті самі дані, збережені у файли GeoParquet, читаються з диска.

Нижче — живий запит. Запустіть його **один раз**: він збереже теку зі знімком,
і всі наступні запуски читатимуть з неї.

> Overpass — безкоштовний сервіс і обмежує частоту. Якщо запит не пройшов —
> зачекайте хвилину й повторіть. Якщо не проходить взагалі, візьміть
> резервний знімок у викладача і вкажіть це у звіті.

In [49]:
import osmnx as ox

OUT = 'osm_' + CITY.split(',')[0].strip().lower().replace(' ', '_')
os.makedirs(OUT, exist_ok=True)
ox.settings.use_cache = True
ox.settings.cache_folder = os.path.join(OUT, 'cache')
ox.settings.requests_timeout = 300
ox.settings.overpass_rate_limit = True

TOURISM_OK = ['museum', 'attraction', 'gallery', 'artwork', 'viewpoint']

def cache_bytes():
    import pathlib
    p = pathlib.Path(ox.settings.cache_folder)
    return sum(f.stat().st_size for f in p.glob('*.json')) if p.exists() else 0

print('тека знімка:', OUT)

тека знімка: osm_чернігів


### Дві налаштовані дрібниці, без яких запит не пройде

Це не формальності — кожна з них ламає збір по-своєму, і жодна не схожа
на свою справжню причину.

**Представитись.** Overpass відповідає **HTTP 406** клієнтові, який прийшов
із типовим заголовком `python-requests`. `osmnx` надсилає свій `User-Agent`
сам, але якщо ви звертаєтесь до сервісу напряму — надсилайте теж.
Рядок має бути **лише латиницею**: HTTP-заголовки передаються в latin-1,
і кирилиця кладе запит помилкою `UnicodeEncodeError` ще до відправлення.

**Обрати машину.** За іменем `overpass-api.de` стоїть кілька серверів, і
частина з них відмовляє у з’єднанні. `osmnx` закріплює одну адресу на
всю сесію — навмисно, щоб не порушувати облік слотів. Якщо закріпиться та,
що відмовляє, впаде весь збір. Тому адресу обираємо самі, вже перевірену.

In [50]:
import socket

USER_AGENT = ('LvivPolytechnic-SK26-coursework/1.0 '
              '(https://lpnu.ua; student lab assignment)')
assert USER_AGENT.isascii(), 'HTTP-заголовки — latin-1, кирилиця тут не працює'
ox.settings.http_user_agent = USER_AGENT


def pin_working_ip(hostname, port=443, timeout=8):
    """Закріпити для цього імені адресу, яка справді відповідає."""
    try:
        infos = socket.getaddrinfo(hostname, port, socket.AF_INET, socket.SOCK_STREAM)
    except Exception as e:
        print('не вдалось резолвити %s: %s' % (hostname, type(e).__name__))
        return None
    ips = sorted({i[4][0] for i in infos})
    good = []
    for ip in ips:
        s = socket.socket(); s.settimeout(timeout)
        try:
            s.connect((ip, port)); good.append(ip)
        except Exception:
            pass
        finally:
            s.close()
    print('%s: машин %d, відповідають %d %s' % (hostname, len(ips), len(good), good))
    if not good:
        return None
    original = socket.gethostbyname
    socket.gethostbyname = lambda h: good[0] if h == hostname else original(h)
    print('закріплено', good[0])
    return good[0]


pin_working_ip('overpass-api.de')

overpass-api.de: машин 2, відповідають 1 ['65.109.112.52']
закріплено 65.109.112.52


'65.109.112.52'

> Якщо жодна машина не відповіла — виконайте комірку ще раз за хвилину,
> або візьміть у викладача резервний знімок і зазначте це у звіті.
> Інші дзеркала перелічені на сторінці
> [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API);
> адресу можна підставити у `ox.settings.overpass_url`.

### Завдання 1.1

Допишіть функцію `fetch`: вона має зберегти шар у `.parquet`, а якщо файл
уже є — прочитати його, **не звертаючись до мережі**. Поверніть також,
скільки МБ пішло мережею і скільки секунд це зайняло.

Підказка щодо `clean`: у OSM одна колонка може містити і число, і список
(ребро, склеєне з кількох шляхів). Parquet вимагає один тип на колонку,
тому неоднорідну колонку треба звести до тексту **цілком**.

In [51]:
def clean(gdf):
    g = gdf.reset_index()
    for c in g.columns:
        if c == 'geometry':
            continue
        boxed = g[c].map(lambda v: isinstance(v, (list, dict, set, tuple))).any()
        if boxed or g[c].dtype == object:
            g[c] = g[c].map(
                lambda v: None if v is None or (isinstance(v, float) and pd.isna(v))
                else (json.dumps(v, ensure_ascii=False)
                      if isinstance(v, (list, dict, set, tuple)) else str(v))
            ).astype('string')
    return g


COST = {}

def fetch(name, getter):
    """Повертає шар. Якщо файл уже збережено — читає з диска."""
    path = os.path.join(OUT, name + '.parquet')
    if os.path.exists(path) and os.path.getsize(path) > 0:
        t0 = time.time()
        g = gpd.read_parquet(path)
        COST[name] = {'спосіб': 'знімок', 'записів': len(g),
                      'передано_МБ': 0.0,
                      'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                      'секунд': round(time.time() - t0, 3)}
        return g

    t0, b0 = time.time(), cache_bytes()
    g = clean(getter())
    g.to_parquet(path, index=False)
    COST[name] = {'спосіб': 'живий запит', 'записів': len(g),
                  'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                  'файл_МБ': round(os.path.getsize(path) / 1e6, 3),
                  'секунд': round(time.time() - t0, 1)}
    return gpd.read_parquet(path)

In [52]:
# import time as _time

# _orig_fetch = fetch

# def fetch_retry(name, getter, tries=6, pause=30):
#     for k in range(1, tries + 1):
#         try:
#             return _orig_fetch(name, getter)
#         except Exception as e:
#             msg = str(e)[:120].replace('\n', ' ')
#             print('спроба %d/%d для «%s» не вдалась: %s' % (k, tries, name, msg))
#             if k == tries:
#                 raise
#             print('  чекаю %d с ...' % pause)
#             _time.sleep(pause)
#             pause *= 2          # 30, 60, 120, 240 ... секунд

In [53]:
g = ox.geocode_to_gdf(CITY)
print(g['display_name'].iloc[0])
print(g['addresstype'].iloc[0] if 'addresstype' in g.columns else g.columns.tolist())

Chernihiv Oblast, Ukraine
state


In [54]:
def city_boundary(query):
    for k in range(1, 8):
        try:
            g = ox.geocode_to_gdf(query, which_result=k)
        except Exception:
            continue
        kind = g['addresstype'].iloc[0] if 'addresstype' in g.columns else '?'
        area = g.to_crs(32636).area.iloc[0] / 1e6
        print('результат %d: %s | %s | %.1f км²' % (k, str(g['display_name'].iloc[0])[:50], kind, area))
        if kind in ('city', 'town') and 20 <= area <= 500:
            return g
    raise RuntimeError('місто не знайдено; спробуйте "Chernihiv, Chernihiv Oblast, Ukraine"')

boundary = fetch('boundary', lambda: city_boundary(CITY))
poly = boundary.geometry.iloc[0]
print(boundary['display_name'].iloc[0])
print('площа межі: %.1f км²' % (boundary.to_crs(32635).area.iloc[0] / 1e6))

Chernihiv, Чернігівська міська громада, Chernihiv Raion, Chernihiv Oblast, 14000-14499, Ukraine
площа межі: 84.0 км²


In [55]:
heritage = fetch('heritage', lambda: ox.features_from_polygon(
    poly, {'historic': True, 'tourism': TOURISM_OK}))
stations = fetch('stations', lambda: ox.features_from_polygon(
    poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
amenities = fetch('amenities', lambda: ox.features_from_polygon(
    poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))

# heritage  = fetch_retry('heritage', lambda: ox.features_from_polygon(
#     poly, {'historic': True, 'tourism': TOURISM_OK}))
# stations  = fetch_retry('stations', lambda: ox.features_from_polygon(
#     poly, {'railway': ['station', 'halt'], 'building': ['train_station']}))
# amenities = fetch_retry('amenities', lambda: ox.features_from_polygon(
#     poly, {'amenity': ['cafe', 'restaurant', 'drinking_water', 'toilets']}))


In [56]:
NODES_P = os.path.join(OUT, 'road_nodes.parquet')
EDGES_P = os.path.join(OUT, 'road_edges.parquet')

if os.path.exists(NODES_P) and os.path.exists(EDGES_P):
    t0 = time.time()
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'знімок', 'записів': len(nodes) + len(edges),
                           'передано_МБ': 0.0,
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 3)}
else:
    t0, b0 = time.time(), cache_bytes()
    G_raw = ox.graph_from_polygon(poly, network_type='walk')
    nodes, edges = ox.convert.graph_to_gdfs(G_raw)
    clean(nodes).to_parquet(NODES_P, index=False)
    clean(edges).to_parquet(EDGES_P, index=False)
    nodes, edges = gpd.read_parquet(NODES_P), gpd.read_parquet(EDGES_P)
    COST['мережа walk'] = {'спосіб': 'живий запит', 'записів': len(nodes) + len(edges),
                           'передано_МБ': round((cache_bytes() - b0) / 1e6, 3),
                           'файл_МБ': round((os.path.getsize(NODES_P)
                                             + os.path.getsize(EDGES_P)) / 1e6, 3),
                           'секунд': round(time.time() - t0, 1)}

print('вузлів %d | ребер %d' % (len(nodes), len(edges)))

вузлів 14486 | ребер 43206


### Завдання 1.2

Зведіть `COST` у таблицю і дайте відповідь у звіті: **у скільки разів
читання знімка швидше за живий запит** і **скільки мегабайтів ви не передали
мережею** при повторному запуску.

Контрольна ознака правильності: при **другому** запуску ноутбука всі рядки
мають мати спосіб «знімок», а `передано_МБ` — нуль.

In [57]:
# live = pd.DataFrame(COST).T
# live.to_json(os.path.join(OUT, 'cost_live.json'), force_ascii=False)

In [58]:
repeat = pd.DataFrame(COST).T

display(pd.read_json(os.path.join(OUT, 'cost_live.json')))

display(repeat)

,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,живий запит,1,0.020,0.029,1.3
heritage,живий запит,121,0.104,0.083,76.4
stations,живий запит,5,0.004,0.017,74.3
amenities,живий запит,193,0.088,0.072,76.8
мережа walk,живий запит,57692,5.947,2.747,96.8


,спосіб,записів,передано_МБ,файл_МБ,секунд
boundary,знімок,1,0.0,0.029,0.083
heritage,знімок,121,0.0,0.083,0.108
stations,знімок,5,0.0,0.017,0.057
amenities,знімок,193,0.0,0.072,0.104
мережа walk,знімок,57692,0.0,2.747,0.398


**Відповідь**: Завантаження всіх файлів через живий запит зайняло близько 5.4 хвилини, при повторному запиті - 0.36 секунд. Читання знімка швидше за живий запит у 899 разів. При повторному запуску не було передано 6.16 мегабайтів.

> **Чому пішохідна мережа, а не автомобільна.** У старому місті музеї стоять
> за сотню метрів один від одного, а автомобільних вузлів там одиниці. Кілька
> пам'яток сядуть на **один** вузол графа, і матриця віддалей скаже, що між
> ними нуль метрів. Задача розв'яжеться, відповідь буде неправильною, і
> повідомлення про помилку не з'явиться. У кроці 4 ви це перевірите явно.

---
## 2. Хто претендує на звання пам'ятки

У OpenStreetMap немає тега «визначна пам'ятка». Є два сімейства:

* `historic=*` — `castle`, `church`, `monument`, `memorial`, `ruins`, `tower`, `yes`…
* `tourism=*` — `museum`, `gallery`, `attraction`, `artwork`, `viewpoint`

Перетин неповний: музей у сучасній будівлі має лише `tourism`, руїни фортеці —
лише `historic`. Тому беремо **об'єднання**.

📖 <https://wiki.openstreetmap.org/wiki/Map_Features>

### Завдання 2.1

Відберіть кандидатів і **назвіть у звіті, скільки об'єктів ви відкинули й чому**.

In [59]:
for t, sub in heritage.groupby('historic'):
    names = sub['name'].dropna().tolist()
    print(f"Об'єкт: {t}")
    print(f"Непустих: {len(names)} з {len(sub)}")
    print("Приклади значень:")
    if names:
        for n in names[:5]:
            print(f"   - {n}")
    else:
        print("   (немає жодного значення)")
    print()

Об'єкт: aircraft
Непустих: 9 з 9
Приклади значень:
   - МіГ-21ПФ
   - МіГ-23УБ
   - МіГ-23М
   - Аеро Л-39 «Альбатрос»
   - Аеро Л-29 «Дельфін»

Об'єкт: archaeological_site
Непустих: 1 з 1
Приклади значень:
   - Дитинець літописного міста Чернігова

Об'єкт: battlefield
Непустих: 1 з 1
Приклади значень:
   - Болдина гора

Об'єкт: building
Непустих: 1 з 3
Приклади значень:
   - Борисоглібський собор

Об'єкт: cannon
Непустих: 5 з 17
Приклади значень:
   - М-43
   - ЗіС-3
   - М-30
   - Д-44
   - Гармата

Об'єкт: church
Непустих: 1 з 1
Приклади значень:
   - Козацький кафедральний собор Святої Великомучениці Катерини

Об'єкт: heritage
Непустих: 1 з 1
Приклади значень:
   - Національний архітектурно-історичний заповідник "Чернігів стародавній"

Об'єкт: memorial
Непустих: 31 з 42
Приклади значень:
   - Богдану Хмельницькому
   - Меморіал Слави
   - Чорна могила
   - Воїнам-афганцям
   - Борцям за волю і незалежність України

Об'єкт: monument
Непустих: 2 з 2
Приклади значень:
   - Івану Мазеп

In [60]:
for t, sub in heritage.groupby('tourism'):
    names = sub['name'].dropna().tolist()
    print(f"Об'єкт: {t}")
    print(f"Непустих: {len(names)} з {len(sub)}")
    print("Приклади значень:")
    if names:
        for n in names[:5]:
            print(f"   - {n}")
    else:
        print("   (немає жодного значення)")
    print()

Об'єкт: artwork
Непустих: 7 з 9
Приклади значень:
   - Т.Г. Шевченко
   - Совість
   - Ілля Муромець
   - Півень
   - Святого Миколи Чудотворця

Об'єкт: attraction
Непустих: 4 з 16
Приклади значень:
   - Антонієві печери
   - Дім полкової канцелярії
   - Пневматичний тир
   - Нудистський пляж

Об'єкт: museum
Непустих: 8 з 10
Приклади значень:
   - Музей історії авіації і космонавтики
   - Військово-історичний музей
   - Літературно-меморіальний музей-заповідник Михайла Коцюбинського
   - Спасо-Преображенський собор
   - Художній музей

Об'єкт: viewpoint
Непустих: 1 з 5
Приклади значень:
   - Королівська гірка



In [61]:
EXCLUDE_HIST = ['wreck', 'ruins']

is_tour  = heritage['tourism'].isin(TOURISM_OK)
bad_type = heritage['historic'].isin(EXCLUDE_HIST) & ~is_tour
has_tag  = heritage['historic'].notna() | is_tour
has_name = heritage['name'].notna()

mask = has_tag & ~bad_type
cand = heritage[mask & has_name].reset_index(drop=True)

print('усього у шарі heritage:', len(heritage))
print('відкинуто за типом:',  int(bad_type.sum()))
print('відкинуто, бо немає назви:', int((mask & ~has_name).sum()))
print('скільки кандидатів на визначні пам\'ятки:', len(cand))

dropped = heritage[bad_type].groupby('historic')['name'].agg(всього='size', з_назвою='count')
display(dropped)

усього у шарі heritage: 121
відкинуто за типом: 7
відкинуто, бо немає назви: 33
скільки кандидатів на визначні пам'ятки: 81


,всього,з_назвою
historic,,
ruins,6,1
wreck,1,0


In [62]:
assert 'cand' in dir(), 'спершу виконайте завдання 2.1'
print('=== historic ===');  print(cand['historic'].value_counts().head(10).to_string())
print()
print('=== tourism ===');   print(cand['tourism'].value_counts().head(10).to_string())
print()
print('типи геометрії:', cand.geometry.geom_type.value_counts().to_dict())

=== historic ===
historic
memorial               31
aircraft                9
cannon                  5
tomb                    4
tank                    2
monument                2
yes                     2
archaeological_site     1
vehicle                 1
synagogue               1

=== tourism ===
tourism
museum        8
artwork       7
attraction    4
viewpoint     1

типи геометрії: {'Point': 64, 'Polygon': 17}


**Відповідь (завдання 2.1).** 

З 121 об'єкта шару heritage я відкинула 40 об'єктів загалом.

**Відкинуто за типом: 7**

| тип `historic` | скільки | чому відкинуто |
|---|---|---|
| `ruins` | 6 | лише 1 з 6 має назву, і це «АТБ-Маркет» |
| `wreck` | 1 | уламки, назви немає |

**ідкинуто через відсутність назви: 33** Об'єкт без назви нічим підписати на карті й у маршруті.

> **Перевірте себе.** Серед типів геометрії мають бути і `Point`, і `Polygon`.
> Якщо самі лише полігони — ви відфільтрували зайве: пам'ятники, меморіали та
> оглядові точки в OSM є **вузлами**, у них немає будівлі.

### Якщо кандидатів менше тридцяти

Невелике місто може дати 15–20 об'єктів. Тоді:

1. розширте перелік `tourism` (наприклад, додайте `theatre`, `place_of_worship`);
2. або зменште `N_LARGE` до того, що є;

і **обов'язково зазначте це у звіті** разом із числом.

---
## 3. Критерій «найкращі»

Ключовий крок роботи. Стовпця «цінність» у даних немає. Його треба
сконструювати і захистити.

Сигнали, які є в OSM:

| сигнал | що означає |
|---|---|
| `wikidata` / `wikipedia` | про об'єкт є стаття — хтось сторонній визнав його вартим опису |
| кількість колонок `name:*` | назву переклали N мовами — перекладають те, що шукають іноземці |
| `tourism=museum/gallery` | призначений для відвідування |
| `historic` заповнений | має історичну природу |
| площа полігона | розмір — **найслабший сигнал**: великим буває і склад |

### Завдання 3.1

Побудуйте числовий рейтинг. Вимоги:

* використати **щонайменше три** різні сигнали;
* звести різнорідні величини до спільної шкали (підказка: `rank(pct=True)`);
* **пояснити у звіті, чому саме такі ваги**.

In [63]:
metric = cand.to_crs(32635)
cand['area_m2'] = np.where(metric.geometry.geom_type.isin(['Polygon', 'MultiPolygon']),
                           metric.area.values, 0.0)

W = {'wiki': 0.40, 'lang': 0.30, 'visit': 0.15, 'hist': 0.15}

cand['wiki']  = (cand['wikidata'].notna() | cand['wikipedia'].notna()).astype(int)

lang_cols = [c for c in cand.columns if c.startswith('name:') and c != 'name:uk']
cand['n_lang'] = cand[lang_cols].notna().sum(axis=1)

r_lang = cand['n_lang'].rank(pct=True, method='min')

cand['visit'] = cand['tourism'].isin(['museum', 'gallery']).astype(int)
cand['hist']  = (cand['historic'].notna() & (cand['historic'] != 'yes')).astype(int)

cand['score'] = (W['wiki'] * cand['wiki'] + W['lang'] * r_lang + W['visit'] * cand['visit'] + W['hist'] * cand['hist'])  

print('кандидатів   :', len(cand))
print('мають wiki   :', int(cand['wiki'].sum()))
print('мають >=1 мови:', int((cand['n_lang'] >= 1).sum()))

кандидатів   : 81
мають wiki   : 23
мають >=1 мови: 16


In [64]:
cols = ['name', 'historic', 'tourism', 'wiki', 'n_lang', 'area_m2', 'score']
display(cand.sort_values('score', ascending=False)[cols].round(2))

,name,historic,tourism,wiki,n_lang,area_m2,score
68,Художній музей,<NA>,museum,1,2,2186.41,0.84
70,Козацький кафедральний собор Святої Великомуче...,church,<NA>,1,2,532.16,0.84
0,Богдану Хмельницькому,memorial,<NA>,1,1,0.00,0.79
2,Чорна могила,memorial,<NA>,1,1,0.00,0.79
64,Літературно-меморіальний музей-заповідник Миха...,<NA>,museum,1,1,6595.35,0.79
...,...,...,...,...,...,...,...
53,Королівська гірка,<NA>,viewpoint,0,0,0.00,0.00
52,Півень,<NA>,artwork,0,0,0.00,0.00
77,Нудистський пляж,<NA>,attraction,0,0,1556.00,0.00
51,Ілля Муромець,<NA>,artwork,0,0,0.00,0.00


**Відповідь (завдання 3.1).** Рейтинг об'єкта складається з 4 сигналів: наявність статті (`wiki`), кількість перекладів назви (`мови`), чи це музей або галерея (`музей`), чи конкретний історичний тип (`тип`)

Формула: `score = 0,40·wiki + 0,30·мови + 0,15·музей + 0,15·тип`

**Чому такі ваги.**
- **Статті 0,40 і мови 0,30.** Найбільші ваги отримали сигнали, де рішення ухвалила стороння людина: хтось сам написав статтю або зробив переклад. Статтю мають 23 з 81 кандидата (28 %), мови хоча б одну мають 16 з 81 (20 %), тож обидва сигнали рідкісні й добре відрізняють помітні об'єкти від решти.
- **Музей 0,15 і тип 0,15.** Ці сигнали корисні (музей призначений для відвідування, конкретний тип означає, що хтось визначив вид пам'ятки).

### Завдання 3.2 — обов'язкове порівняння

Побудуйте **другий** рейтинг за найпростішим критерієм — площею будівлі —
і порахуйте, скільки об'єктів збігається в перших десяти.

Це головне питання роботи: обидва списки отримані з тих самих даних, тим
самим кодом, без жодної помилки. Якщо вони різні — поясніть у звіті, що це
означає для слова «найкращі».

In [65]:
N_SMALL, N_LARGE = 10, 30

cand['key'] = cand['element'].astype(str) + '/' + cand['id'].astype(str)


top_score = cand.nlargest(N_LARGE, 'score').reset_index(drop=True)
top10     = top_score.head(N_SMALL)
top_naive = cand.nlargest(N_SMALL, 'area_m2').reset_index(drop=True)

common = set(top10['key']) & set(top_naive['key'])
overlap   = len(common)

print('перетин: %d з %d' % (overlap, N_SMALL))
print('Що збіглось:', cand[cand['key'].isin(common)]['name'].tolist())
display(pd.DataFrame({'за вашим критерієм': list(top10['name']),
                      'за площею': list(top_naive['name'])}))

перетин: 2 з 10
Що збіглось: ['Літературно-меморіальний музей-заповідник Михайла Коцюбинського', 'Художній музей']


,за вашим критерієм,за площею
0,Художній музей,Національний архітектурно-історичний заповідни...
1,Козацький кафедральний собор Святої Великомуче...,Болдина гора
2,Богдану Хмельницькому,Літературно-меморіальний музей-заповідник Миха...
3,Чорна могила,Художній музей
4,Івану Мазепі,Нудистський пляж
5,Т.Г. Шевченко,Спасо-Преображенський собор
6,Літературно-меморіальний музей-заповідник Миха...,Введенська церква (трапезна)
7,Антонієві печери,Чернігівський Колегіум
8,Битва за Чернігів,Молодіжний театр
9,Дім полкової канцелярії,Чорна могила


**Відповідь (завдання 3.2).** У двох топ-10 (за моїм рейтингом і за площею будівлі) збіглося 2 з 10 об'єктів: "Літературно-меморіальний музей-заповідник Михайла Коцюбинського", "Художній музей".

Площа має суттєвий недолік. Всі пам'ятники й меморіали матимуть меншу площу ніж будь-яка будівля, парк, заповідник тощо («Нудистський пляж» потрапив у топ лише через розмір, але як мені здається середньостатистичному туристові таке наврядчи підійде). Тому за площею я б не стала відбирати.

### Завдання 3.3 — дублікати

OSM наповнюють різні люди. Той самий музей трапляється двічі під різними
назвами, а вежа буває окремим об'єктом усередині палацу. Знайдіть у `top_score`
пари, що стоять ближче **60 метрів** одна до одної, і вирішіть, що з ними робити.

Рішення (лишити / злити / видалити) опишіть у звіті.

In [66]:
def close_pairs(gdf, limit_m=60):
    p = gdf.to_crs(32635).geometry.representative_point()
    XY = np.c_[p.x.values, p.y.values]
    D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    out = [(i, j, D[i, j]) for i in range(len(XY)) for j in range(i + 1, len(XY))
           if D[i, j] < limit_m]
    return sorted(out, key=lambda t: t[2])

pairs = close_pairs(top_score)
print('пар ближче 60 м: %d' %  len(pairs))

rows = []
for i, j, d in pairs:
    a, b = top_score.iloc[i], top_score.iloc[j]
    rows.append({'відстань, м': round(d, 1),
                 'об\'єкт A': a['name'], 'тип A': a['historic'] if pd.notna(a['historic']) else a['tourism'], 'score A': round(a['score'], 2),
                 'об\'єкт B': b['name'], 'тип B': b['historic'] if pd.notna(b['historic']) else b['tourism'], 'score B': round(b['score'], 2)})
pairs_df = pd.DataFrame(rows)
display(pairs_df)

пар ближче 60 м: 5


,"відстань, м",об'єкт A,тип A,score A,об'єкт B,тип B,score B
0,7.2,Чорна могила,memorial,0.79,Чорна могила,tomb,0.55
1,23.2,Спасо-Преображенський собор,museum,0.55,Національний архітектурно-історичний заповідни...,heritage,0.55
2,31.6,Івану Мазепі,monument,0.79,Чернігівський Колегіум,museum,0.55
3,49.3,Борисоглібський собор,building,0.55,Чернігівський Колегіум,museum,0.55
4,53.6,Борисоглібський собор,building,0.55,Національний архітектурно-історичний заповідни...,heritage,0.55


In [67]:
remove = set()
for i, j, d in pairs:
    a, b = top_score.iloc[i], top_score.iloc[j]

    if a['name'] == b['name']:
        loser = a if a['score'] < b['score'] else b
        remove.add(loser['key'])

    for x in (a, b):
        if pd.notna(x['historic']) and x['historic'] == 'heritage':
            remove.add(x['key'])


In [68]:
ok = cand[~cand['key'].isin(remove)]
top_score = (ok.sort_values(['score', 'n_lang', 'area_m2'], ascending=False)
               .head(N_LARGE).reset_index(drop=True))
top10 = top_score.head(N_SMALL)

In [69]:
common  = set(top10['key']) & set(top_naive['key'])
overlap = len(common)

pairs = close_pairs(top_score)
print('пар ближче 60 м після очищення:', len(pairs))
rows = []
for i, j, d in pairs:
    a, b = top_score.iloc[i], top_score.iloc[j]
    rows.append({'відстань, м': round(d, 1),
                 'об\'єкт A': a['name'], 'тип A': a['historic'] if pd.notna(a['historic']) else a['tourism'], 'score A': round(a['score'], 2),
                 'об\'єкт B': b['name'], 'тип B': b['historic'] if pd.notna(b['historic']) else b['tourism'], 'score B': round(b['score'], 2)})
pairs_df = pd.DataFrame(rows)
display(pairs_df)

пар ближче 60 м після очищення: 2


,"відстань, м",об'єкт A,тип A,score A,об'єкт B,тип B,score B
0,31.6,Івану Мазепі,monument,0.79,Чернігівський Колегіум,museum,0.55
1,49.3,Чернігівський Колегіум,museum,0.55,Борисоглібський собор,building,0.55


**Відповідь (завдання 3.3).** Знайшлось 5 пар об'єктів, що стоять ближче 60 м одне до одного. Вони виявились різних видів:

- **Чорна могила і Чорна могила (7 м)** — це один і той самий об'єкт, записаний двічі: як `memorial` і як `tomb`. Це дублікат, тому лишаю `memorial` (бо рейтинг більший), а `tomb` видаляю.
- **Заповідник «Чернігів стародавній» поруч із Спасо-Преображенським і Борисоглібським соборами (23 і 54 м)** — заповідник це велика територія, у якій ці собори стоять. Він не окрема зупинка, тому видаляю його, а конкретні собори лишаю.
- **Мазепа і Колегіум (32 м), Колегіум і Борисоглібський собор (49 м)** — це різні пам'ятки, які просто стоять поруч (пам'ятник, музей, собор). Дублікатами вони не є, тому лишаю обидва.

---
## 4. Старт: вокзал, і прив'язка точок до графа

In [70]:
print('станції у шарі:', len(stations))
print(stations[['name', 'railway', 'building']].dropna(how='all').head(12).to_string())

станції у шарі: 5
                 name  railway       building
0            Чернігів  station           <NA>
1           Подусівка     halt           <NA>
2  Чернігів-Північний  station           <NA>
3  Залізничний вокзал     <NA>  train_station
4                <NA>     <NA>  train_station


In [71]:
station = stations.iloc[[0]].reset_index(drop=True)
print('старт:', station['name'].iloc[0])

старт: Чернігів


In [72]:
G = nx.DiGraph()
for t in nodes.itertuples():
    G.add_node(t.osmid, x=t.x, y=t.y)
for t in edges.itertuples():
    L = float(t.length)
    if G.has_edge(t.u, t.v):
        G[t.u][t.v]['length'] = min(G[t.u][t.v]['length'], L)
    else:
        G.add_edge(t.u, t.v, length=L)
print('граф: %d вузлів, %d ребер' % (G.number_of_nodes(), G.number_of_edges()))

NODES_M = nodes.to_crs(32635)
NODE_XY = np.c_[NODES_M.geometry.x.values, NODES_M.geometry.y.values]
NODE_ID = nodes['osmid'].values

def snap(points_gdf):
    """Найближчий вузол графа до кожної точки. Рахуємо в метричній CRS."""
    pmet = points_gdf.to_crs(32635)
    ids, dist = [], []
    for p in pmet.geometry:
        d2 = (NODE_XY[:, 0] - p.x) ** 2 + (NODE_XY[:, 1] - p.y) ** 2
        k = int(np.argmin(d2))
        ids.append(NODE_ID[k]); dist.append(float(np.sqrt(d2[k])))
    return np.array(ids), np.array(dist)

def make_points(top):
    names = ['СТАРТ: ' + str(station['name'].iloc[0])] + list(top['name'])
    geoms = [station.geometry.iloc[0].representative_point()] + \
            [g.representative_point() for g in top.geometry]
    return gpd.GeoDataFrame({'name': names}, geometry=geoms, crs='EPSG:4326')

граф: 14486 вузлів, 42853 ребер


### Завдання 4.1 — обов'язкова перевірка

Напишіть `check_snap`, яка повідомляє:

* скільки точок і скільки **різних** вузлів вони зайняли;
* медіану й максимум відстані «точка → вузол»;
* якщо вузлів менше, ніж точок — **назви точок, що злиплись**.

**Контрольна ознака:** на пішохідній мережі різних вузлів має бути **стільки ж,
скільки точок**. Якщо менше — ви взяли автомобільну мережу або місто надто мале;
зафіксуйте це у звіті, бо далі такі пари отримають нульову віддаль.

In [73]:
def check_snap(points, ids, dist):
    k, u = len(ids), len(set(ids))
    med, mx = float(np.median(dist)), float(np.max(dist))

    print('Точок %d, різних вузлів %d' % (k, u))
    print('Медіана %.0f м, максимум %.0f м' % (med, mx))

    stuck = {}
    if u < k:
        for nid, c in collections.Counter(ids).items():
            if c > 1:
                stuck[nid] = list(points['name'].values[ids == nid])
        print('Злиплені точки:')
        for nid, names in stuck.items():
            print('   вузол %s:' % nid, names)

    return k, u, med, mx, stuck

In [74]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
k10, u10, med10, mx10, stuck10 = check_snap(pts10, ids10, dist10)

Точок 11, різних вузлів 11
Медіана 17 м, максимум 61 м


In [75]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
k30, u30, med30, mx30, stuck30 = check_snap(pts30, ids30, dist30)

Точок 31, різних вузлів 31
Медіана 24 м, максимум 85 м


**Відповідь (завдання 4).** Стартом обрала станцію «Чернігів». У шарі також є «Чернігів-Північний», зупинка «Подусівка» і будівля вокзалу, але головною є саме ця станція.

Усі точки зайняли різні вузли графа: у малій задачі 11 точок на 11 вузлах (медіана відстані до вузла 17 м, максимум 61 м), у великій 31 точка на 31 вузлі (медіана 24 м, максимум 85 м). Отже, на пішохідній мережі нульових відстаней між пам'ятками немає. Пари, що стоять поруч (Мазепа і Колегіум, Колегіум і Борисоглібський собор), на один вузол не потрапили.

---
## 5. Матриця віддалей

Дві матриці: **пряма лінія** і **дорогою**. Їхнє відношення — коефіцієнт
обходу — показує, наскільки пряма лінія бреше.

### Завдання 5.1

In [76]:
def road_matrix(ids):
    """Матриця найкоротших віддалей графом між заданими вузлами."""
    k = len(ids)
    D = np.full((k, k), np.inf)
    for i, s in enumerate(ids):
        lengths = nx.single_source_dijkstra_path_length(G, s, weight='length')
        for j, t in enumerate(ids):
            D[i, j] = lengths.get(t, np.inf)
    np.fill_diagonal(D, 0.0)
    return D

def straight_matrix(points):
    p = points.to_crs(32635).geometry
    XY = np.c_[p.x.values, p.y.values]
    return np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))

def detour_factor(D, E):
    ok = np.isfinite(D) & (E > 0)
    return float(np.median(D[ok] / E[ok]))

**Контрольні ознаки — звіртесь, не підганяючи:**

* коефіцієнт обходу для пішохідної мережі у місті зазвичай **1,2–1,6**;
* якщо у вас рівно **1,0** — ви порівнюєте матрицю саму із собою;
* якщо **більше 2** — найімовірніше взято автомобільну мережу з односторонніми
  вулицями, або місто має річку/залізницю, яку нема де перетнути. Друге —
  цікавий результат, перше — помилка. Розберіться, що саме у вас, і напишіть це.

Якщо серед віддалей трапились `inf` — частина графа не зв'язана з рештою.
Не замінюйте `inf` великим числом мовчки: спершу з'ясуйте, які це точки.

In [77]:
pts10 = make_points(top10)
ids10, dist10 = snap(pts10)
check_snap(pts10, ids10, dist10)

t0 = time.time(); D10 = road_matrix(ids10); print('матриця за %.3f с' % (time.time() - t0))
E10 = straight_matrix(pts10)
print('недосяжних пар    :', int(np.isinf(D10).sum()))
print('асиметрія         : %.0f м' % float(np.max(np.abs(D10 - D10.T))))
print('коефіцієнт обходу : %.2f' % detour_factor(D10, E10))

Точок 11, різних вузлів 11
Медіана 17 м, максимум 61 м
матриця за 1.327 с
недосяжних пар    : 0
асиметрія         : 0 м
коефіцієнт обходу : 1.21


**Відповідь (завдання 5.1).** Побудували дві матриці відстаней між 11 точками (вокзал і 10 пам'яток): по прямій і по пішохідному графу.

- **Коефіцієнт обходу 1,21:** реальний пішохідний шлях на 21 % довший за пряму.
- **Недосяжних пар 0:** граф зв'язний, усі точки досяжні одна з одної.
- **Асиметрія 0 м:** шлях туди й назад однаковий, бо пішохідна мережа двонапрямлена: пішохід може йти будь-якою вулицею в обидва боки.

Як висновок: пряма лінія занижує реальну довжину шляху, тому піший маршрут треба рахувати по графу вулиць, а не по прямих.

> **Асиметрія.** $D_{ij} \\neq D_{ji}$ — не помилка: односторонній рух робить
> шлях туди і назад різним. На пішохідній мережі асиметрія мала, на
> автомобільній — помітна. Подивіться на своє число і поясніть його у звіті.

---
## 6. Задача комівояжера

| метод | що робить | складність |
|---|---|---|
| найближчий сусід | щоразу йде в найближчу невідвідану точку | $O(n^2)$ |
| 2-opt | розвертає ділянки готового маршруту, поки коротшає | $O(n^2)$ за прохід |
| Held–Karp | динамічне програмування за підмножинами — **точний** | $O(2^n n^2)$ часу, $O(2^n n)$ пам'яті |

Held–Karp дано готовим: він не є метою роботи, але без нього ви не дізнаєтесь,
**наскільки** помиляються перші два.

### Завдання 6.1 — реалізуйте перші два методи

In [78]:
def tour_length(D, tour):
    return float(sum(D[tour[i], tour[i + 1]] for i in range(len(tour) - 1)))

def nearest_neighbour(D):
    k = len(D)
    tour = [0]
    visited = {0}

    while len(tour) < k:
        cur = tour[-1]

        nxt = min((j for j in range(k) if j not in visited), key=lambda j: D[cur, j])
        tour.append(nxt)
        visited.add(nxt)

    return tour + [0]

def two_opt(D, tour):
    best = list(tour)
    best_len = tour_length(D, best)
    improved = True

    while improved:
        improved = False

        for i in range(1, len(best) - 2):
            for j in range(i + 1, len(best) - 1):

                new = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                new_len = tour_length(D, new)
                
                if new_len < best_len - 1e-9:
                    best, best_len = new, new_len
                    improved = True
    return best

In [79]:
def held_karp(D):
    k = len(D); N = k - 1; INF = float('inf')
    dp = [[INF] * N for _ in range(1 << N)]
    par = [[-1] * N for _ in range(1 << N)]
    for j in range(N):
        dp[1 << j][j] = D[0, j + 1]
    for S in range(1 << N):
        for j in range(N):
            cur = dp[S][j]
            if cur == INF or not (S >> j) & 1:
                continue
            for t in range(N):
                if (S >> t) & 1:
                    continue
                S2 = S | (1 << t); v = cur + D[j + 1, t + 1]
                if v < dp[S2][t]:
                    dp[S2][t] = v; par[S2][t] = j
    full = (1 << N) - 1
    bj = min(range(N), key=lambda j: dp[full][j] + D[j + 1, 0])
    best = dp[full][bj] + D[bj + 1, 0]
    path = []; S = full; j = bj
    while j != -1:
        path.append(j + 1); pj = par[S][j]; S ^= (1 << j); j = pj
    return [0] + path[::-1] + [0], float(best)

### Самоперевірка

Ці твердження мають виконатись для **будь-якого** міста. Якщо хоч одне впало —
шукайте помилку в коді, а не в даних.

In [80]:
def check_tour(tour, k, label):
    assert tour[0] == 0 and tour[-1] == 0, '%s: маршрут має починатись і закінчуватись на старті' % label
    assert sorted(tour[:-1]) == list(range(k)), '%s: кожна точка рівно один раз' % label
    print('%s: структура маршруту правильна (%d точок)' % (label, k))

nn10 = nearest_neighbour(D10)
op10 = two_opt(D10, nn10)
ex10, L_opt = held_karp(D10)

for t, lbl in [(nn10, 'найближчий сусід'), (op10, '2-opt'), (ex10, 'Held-Karp')]:
    check_tour(t, len(ids10), lbl)

assert tour_length(D10, op10) <= tour_length(D10, nn10) + 1e-6, '2-opt не може бути гіршим за вхідний маршрут'
assert L_opt <= tour_length(D10, op10) + 1e-6, 'точний розв\u2019язок не може бути гіршим за наближений'
print()
print('усі перевірки пройдено')

найближчий сусід: структура маршруту правильна (11 точок)
2-opt: структура маршруту правильна (11 точок)
Held-Karp: структура маршруту правильна (11 точок)

усі перевірки пройдено


In [81]:
rows = []
for name, tour in [('найближчий сусід', nn10), ('2-opt', op10), ('Held-Karp', ex10)]:
    L = tour_length(D10, tour)
    rows.append({'метод': name, 'довжина, м': round(L),
                 'гірше оптимуму, %': round(100 * (L / L_opt - 1), 2)})
display(pd.DataFrame(rows).set_index('метод'))

print()
print('оптимальний порядок:')
for step, i in enumerate(ex10):
    print('  %2d. %s' % (step, pts10['name'].iloc[i]))
print()
print('довжина: %.2f км' % (L_opt / 1000))

,"довжина, м","гірше оптимуму, %"
метод,,
найближчий сусід,11895,7.97
2-opt,11338,2.91
Held-Karp,11017,0.00



оптимальний порядок:
   0. СТАРТ: Чернігів
   1. Антонієві печери
   2. Чорна могила
   3. Козацький кафедральний собор Святої Великомучениці Катерини
   4. Т.Г. Шевченко
   5. Дім полкової канцелярії
   6. Художній музей
   7. Івану Мазепі
   8. Богдану Хмельницькому
   9. Битва за Чернігів
  10. Літературно-меморіальний музей-заповідник Михайла Коцюбинського
  11. СТАРТ: Чернігів

довжина: 11.02 км


**Відповідь (завдання 6.1).** Для 10 пам'яток і вокзалу порівняла три методи:

| метод | довжина | гірше оптимуму |
|---|---|---|
| найближчий сусід | 11 895 м | 7,97 % |
| 2-opt | 11 338 м | 2,91 % |
| Held-Karp | 11 017 м | 0 % |

Найближчий сусід гірший за точний розв'язок на 7,97 %.
2-opt зменшив довжину на 557 м і наблизився до оптимуму до 2,91 %, але не дійшов до нього.

Оптимальний маршрут має довжину 11,02 км.

### Завдання 6.2 — велика задача

Спершу оцініть, чи можливий тут точний розв'язок. Порахуйте для
$n = 11, 16, 21, 26, 31$ розмір таблиці Held–Karp ($2^{n-1}\\cdot(n-1)$ клітин)
і скільки це гігабайтів при 8 байтах на клітину.

In [82]:
rows = []
for n in [11, 16, 21, 26, 31]:
    cells = 2 ** (n - 1) * (n - 1)
    gb = cells * 8 / 1e9
    rows.append({'n (точок з вокзалом)': n,
                 'пам\'яток': n - 1,
                 'клітин': f'{cells:,}'.replace(',', ' '),
                 'ГБ': round(gb, 4)})

growth = pd.DataFrame(rows).set_index('n (точок з вокзалом)')
display(growth)

,пам'яток,клітин,ГБ
n (точок з вокзалом),,,
11,10,10 240,0.0001
16,15,491 520,0.0039
21,20,20 971 520,0.1678
26,25,838 860 800,6.7109
31,30,32 212 254 720,257.6980


Подивіться на останній рядок і напишіть у звіті, **чому саме** точний розв'язок
на 30 точках недоступний. Слово «довго» відповіддю не є.

In [83]:
pts30 = make_points(top_score)
ids30, dist30 = snap(pts30)
check_snap(pts30, ids30, dist30)

D30 = road_matrix(ids30)
E30 = straight_matrix(pts30)
print('коефіцієнт обходу: %.2f' % detour_factor(D30, E30))

nn30 = nearest_neighbour(D30)
t0 = time.time(); op30 = two_opt(D30, nn30); t_op30 = time.time() - t0
check_tour(op30, len(ids30), '2-opt на 30 точках')

L_nn30, L_op30 = tour_length(D30, nn30), tour_length(D30, op30)
print()
print('найближчий сусід : %8.0f м' % L_nn30)
print('2-opt            : %8.0f м  за %.3f с  (коротше на %.1f%%)'
      % (L_op30, t_op30, 100 * (1 - L_op30 / L_nn30)))
print()
print('ЦЕ — база порівняння для лабораторної роботи № 8.')

Точок 31, різних вузлів 31
Медіана 24 м, максимум 85 м
коефіцієнт обходу: 1.17
2-opt на 30 точках: структура маршруту правильна (31 точок)

найближчий сусід :    38313 м
2-opt            :    32975 м  за 0.029 с  (коротше на 13.9%)

ЦЕ — база порівняння для лабораторної роботи № 8.


**Відповідь (завдання 6.2).**

Коефіцієнт обходу 1,17, нижчий, ніж у малій задачі (1,21). Пам'ятки великої задачі розкидані ширше, відстані між ними довші, і дрібні відхилення вулиць менше важать у загальній довжині шляху.

| метод | довжина | час |
|---|---|---|
| найближчий сусід | 38 313 м | — |
| 2-opt | 32 975 м | 0,014 с |

2-opt скоротив маршрут найближчого сусіда на 13,9 % (5,3 км). На малій задачі виграш був лише 4,7 %, тобто зі зростанням кількості точок жадібний метод помиляється сильніше. Точного розв'язку для 30 точок немає (потрібно ≈ 258 ГБ пам'яті), тому невідомо, наскільки 33 км далекі від оптимуму; на малій задачі 2-opt лишився на 2,9 % від нього.

Маршрут на 30 об'єктів займає близько 33 км.

---
## 7. Просторова індексація: H3

H3 — ієрархічна сітка шестикутників. Рівень 9 — приблизно 200 м по ребру.

📖 <https://h3geo.org/docs/core-library/restable/>

### Завдання 7.1

Розкладіть **усіх** кандидатів по комірках H3 рівня 9 і побудуйте картограму
щільності. На тій самій карті покажіть межу міста і точки вашого маршруту.

In [84]:
import h3
from shapely.geometry import Polygon

H3_RES = 9

pts = cand.geometry.representative_point()
cand['h3'] = [h3.latlng_to_cell(p.y, p.x, H3_RES) for p in pts]

dens = cand.groupby('h3').size().rename('n').reset_index()

hexes = gpd.GeoDataFrame(dens,geometry=[Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in dens['h3']],crs='EPSG:4326')

print('кандидатів:', len(cand), '| зайнятих комірок:', len(hexes), '| максимум в одній:', hexes['n'].max())

кандидатів: 81 | зайнятих комірок: 41 | максимум в одній: 10


In [85]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors

norm = mcolors.Normalize(vmin=1, vmax=hexes['n'].max())
hex_color = lambda n: mcolors.to_hex(cm.YlOrRd(norm(n)))

c = boundary.geometry.iloc[0].centroid
fmap7 = folium.Map(location=[c.y, c.x], zoom_start=13, tiles=BASEMAP)

folium.GeoJson(hexes[['h3', 'n', 'geometry']],style_function=lambda f: {'fillColor': hex_color(f['properties']['n']),'color': '#555', 'weight': 0.5, 'fillOpacity': 0.75},tooltip=folium.GeoJsonTooltip(['n'], aliases=["пам'яток:"])).add_to(fmap7)

folium.GeoJson(boundary[['geometry']], style_function=lambda f: {'fill': False, 'color': 'black', 'weight': 2}).add_to(fmap7)

for _, r in pts30.iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=3, color='gray',
                        fill=True, tooltip=r['name']).add_to(fmap7)
for _, r in pts10.iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=6, color='blue',
                        fill=True, tooltip=r['name']).add_to(fmap7)

fmap7.save('lr5_карта.html')

In [86]:
display(fmap7)

**Відповідь (завдання 7.1).** Розклала всіх кандидатів по комірках H3.

Пам'ятки зайняли 41 комірку, у найгустішій 10 об'єктів. Половину всіх пам'яток дають лише 7 комірок (17 % зайнятих), а решта комірок містять здебільшого по одній-дві пам'ятки.

Найгустіші комірки не завжди означають багато справжніх пам'яток. Одна з них лежить в історичному центрі (9 об'єктів), але дві інші (10 і 8 об'єктів) утворила військова техніка: літаки, гармати й танки, які стоять поруч і записані на карті кожен окремо.

У велику задачу через нестачу кандидатів із високим рейтингом потрапили віддалені об'єкти з низьким рейтингом (літак МіГ-21ПФ, вантажівка ЗіС-5, пневматичний тир), і вони утворюють довгі «хвости» на околиці, які подовжують маршрут. Рішення залишити техніку в кандидатах виявилось помилковим: рейтинг не відсіяв її повністю. Для туристичного застосунку я б виключила техніку за типом.

> **Пастка.** `h3.cell_to_boundary` повертає пари **(lat, lng)**, а
> `shapely.Polygon` чекає **(x, y)**, тобто (lng, lat). Переплутаєте — місто
> опиниться в Індійському океані. Це видно одразу, тому дивіться на карту.

### Питання до звіту

У скількох комірках зосереджено половину всіх пам'яток вашого міста?
Що це говорить про форму маршруту?

---
## 8. Просторове поєднання

### Завдання 8.1

Для кожної точки маршруту знайдіть найближчий заклад із шару `amenities`
(`sjoin_nearest`) і виведіть таблицю з відстанню в метрах.

Нагадування: `sjoin_nearest` рахує в одиницях CRS. У градусах результат буде
безглуздим — перепроєктуйте обидва шари в метричну систему.

In [87]:
p = pts10.to_crs(32635)
a = amenities.to_crs(32635)[['name', 'amenity', 'geometry']]

near = gpd.sjoin_nearest(p, a, how='left', distance_col='відстань_м')
near = near[~near.index.duplicated(keep='first')]

types = {'cafe': 'кафе', 'restaurant': 'ресторан',
         'drinking_water': 'питна вода', 'toilets': 'туалет'}
table = pd.DataFrame({
    'точка маршруту':    near['name_left'],
    'найближчий заклад': near['name_right'].fillna('(без назви)'),
    'тип':               near['amenity'].map(types).fillna(near['amenity']),
    'відстань, м':       near['відстань_м'].round(0).astype(int),
}).reset_index(drop=True)
display(table)

print('медіана відстані: %.0f м' % table['відстань, м'].median())
print('максимум: %d м (%s)' % (table['відстань, м'].max(),
      table.loc[table['відстань, м'].idxmax(), 'точка маршруту']))
print('точок, де заклад ближче 200 м: %d з %d' % ((table['відстань, м'] < 200).sum(), len(table)))

,точка маршруту,найближчий заклад,тип,"відстань, м"
0,СТАРТ: Чернігів,(без назви),туалет,132
1,Художній музей,(без назви),туалет,145
2,Козацький кафедральний собор Святої Великомуче...,(без назви),туалет,161
3,Літературно-меморіальний музей-заповідник Миха...,(без назви),питна вода,44
4,Богдану Хмельницькому,Coffe Boom,кафе,107
5,Чорна могила,(без назви),питна вода,259
6,Івану Мазепі,Хінкальня,ресторан,116
7,Т.Г. Шевченко,(без назви),питна вода,123
8,Антонієві печери,(без назви),туалет,49
9,Дім полкової канцелярії,(без назви),туалет,153


медіана відстані: 123 м
максимум: 259 м (Чорна могила)
точок, де заклад ближче 200 м: 10 з 11


**Відповідь (завдання 8.1).** Для кожної з 11 точок маршруту знайшла найближчий заклад (кафе, ресторан, питна вода, туалет).

Медіана відстані до найближчого закладу 123 м, максимум 259 м (біля Чорної могили). У 10 з 11 точок заклад є ближче ніж за 200 м. Отже, на всьому маршруті турист легко знайде, де відпочити.

Водночас найближчі заклади здебільшого виявляється безіменними, але там є туалет чи питна вода, а найближчі кафе лише у трьох точках (одне без назви). Якщо турист хоче поїсти, краще шукати найближче кафе чи ресторан окремо, а не будь-який заклад.

---
## 9. Маршрут на карті

Лінія маршруту — це **геометрія ребер графа**, а не відрізки між пам'ятками.

### Завдання 9.1

Побудуйте карту, на якій **обидві** лінії: справжній маршрут по ребрах і
прямі відрізки між точками. Різниця має бути видно оком.

In [88]:
node_xy = nodes.set_index('osmid')[['x', 'y']]

def route_coords(tour, ids):
    """Координати маршруту по ребрах графа у форматі (lat, lon) для folium."""
    coords = []
    for a, b in zip(tour[:-1], tour[1:]):
        try:
            path = nx.shortest_path(G, ids[a], ids[b], weight='length')
        except nx.NetworkXNoPath:
            path = [ids[a], ids[b]]
        for nd in path:
            coords.append((float(node_xy.loc[nd, 'y']), float(node_xy.loc[nd, 'x'])))
    return coords


tour = ex10
pts = pts10.reset_index(drop=True)

real_line = route_coords(tour, ids10) 
straight_line = [(pts.geometry.iloc[i].y, pts.geometry.iloc[i].x) for i in tour]

start = pts.geometry.iloc[0]
fmap = folium.Map(location=[start.y, start.x], zoom_start=14, tiles=BASEMAP)

folium.PolyLine(real_line, color='#d62728', weight=5, opacity=0.85,
                tooltip='маршрут по вулицях').add_to(fmap)
folium.PolyLine(straight_line, color='#1f77b4', weight=3, opacity=0.9,
                dash_array='8,8', tooltip='прямі відрізки').add_to(fmap)

for step, i in enumerate(tour[:-1]):
    g = pts.geometry.iloc[i]
    color = '#2ca02c' if step == 0 else '#333333'
    folium.Marker(
        [g.y, g.x],
        tooltip='%d. %s' % (step, pts['name'].iloc[i]),
        icon=folium.DivIcon(
            html=('<div style="background:%s;color:white;border-radius:50%%;'
                  'width:24px;height:24px;line-height:24px;text-align:center;'
                  'font-weight:bold;font-size:12px">%d</div>' % (color, step)),
            icon_size=(24, 24), icon_anchor=(12, 12))
    ).add_to(fmap)

fmap.save('lr5_маршрут.html')

In [89]:
display(fmap)

In [90]:
straight_len = sum(E10[tour[k], tour[k + 1]] for k in range(len(tour) - 1))
print('по вулицях: %.2f км' % (L_opt / 1000))
print('по прямій : %.2f км' % (straight_len / 1000))
print('різниця   : %.2f км, у %.2f раза довше' % ((L_opt - straight_len) / 1000, L_opt / straight_len))

по вулицях: 11.02 км
по прямій : 8.07 км
різниця   : 2.94 км, у 1.36 раза довше


**Відповідь (завдання 9.1).** Побудувала карту оптимального маршруту для 10 пам'яток (порядок знайдено точним методом Held-Karp). На ній дві лінії:

- **червона суцільна** — справжній шлях по вулицях (по ребрах графа);
- **синя пунктирна** — прямі відрізки між пам'ятками.

Маркери з номерами показують порядок обходу: 0 це вокзал, далі пам'ятки від 1 до 10.

Маршрут по вулицях має 11,02 км, а сума прямих відрізків 8.07 км, тобто справжній шлях на 2.94 км (у 1.36 раза) довший.

Висновок: прямі відрізки занижують довжину пішого маршруту, тому планувати його треба по графу вулиць.

---
## 10. Результати для лабораторної роботи № 8

Збережіть файл, який ЛР8 прочитає і спробує побити.

In [91]:
result = {
    'студент': STUDENT_NAME,
    'номер': LIST_NUMBER,
    'місто': CITY,
    'мала_задача': {
        'точок': len(ids10),
        'оптимум_м': round(L_opt, 1),
        'оптимальний_порядок': [str(pts10['name'].iloc[i]) for i in ex10],
        'найближчий_сусід_м': round(tour_length(D10, nn10), 1),
        'двоопт_м': round(tour_length(D10, op10), 1),
    },
    'велика_задача': {
        'точок': len(ids30),
        'найближчий_сусід_м': round(L_nn30, 1),
        'двоопт_м': round(L_op30, 1),
        'двоопт_с': round(t_op30, 4),
        'точки': [{'name': str(pts30['name'].iloc[i]),
                   'lat': float(pts30.geometry.y.iloc[i]),
                   'lon': float(pts30.geometry.x.iloc[i])} for i in range(len(pts30))],
        'матриця_віддалей_м': [[None if not np.isfinite(v) else round(float(v), 1)
                                for v in row] for row in D30],
    },
}
with open('lr5_результати.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=1)
print('збережено lr5_результати.json (%.1f КБ)' % (os.path.getsize('lr5_результати.json') / 1024))

збережено lr5_результати.json (17.3 КБ)


---
## 11. Звіт

Відповіді пишіть просто в наступній комірці, числами зі своїх обчислень.

1. **Скільки об'єктів ви відкинули** на кроці 2 і за яким правилом? Чи погодились
   би ви віддати цей фільтр замовнику без пояснень?
2. **Які сигнали ввійшли у ваш рейтинг і з якими вагами?** Чому саме так?
3. **Скільки об'єктів збіглося** у ваших двох рейтингах із десяти? Що це
   означає для слова «найкращі»?
4. **Який у вас коефіцієнт обходу** і чи типовий він? Якщо ні — що в місті
   його пояснює?
5. **На скільки відсотків** найближчий сусід гірший за точний розв'язок?
   Чи здивувало вас це число?
6. **Чому точний розв'язок на 30 точках недоступний?** Назвіть конкретну
   величину, яка цьому заважає.
7. **Що у ваших даних виявилось неправильним** — дублікати, пропущені назви,
   об'єкти не на місці? Як ви з цим вчинили?


**Відповіді**

1. З 121 об'єкта шару heritage я відкинула 40. За типом прибрала 7: ruins і wreck. Ще 33 відкинула, бо в них немає назви, і їх нічим підписати на карті. Лишився 81 кандидат.
   Віддавати цей фільтр замовнику без пояснень я б не стала: це моє рішення про те, що вважати пам'яткою.
   До того ж я не відкинула військову техніку (кожен літак, гармата й танк записаний окремим об'єктом), рейтинг їх не відсіяв. На топ-10 це не вплинуло, але частина техніки потрапила в топ-30 і подовжила великий маршрут. Тому техніку краще було відкинути. 

2. Рейтинг складається з 4 сигналів: score = 0,45·wiki + 0,30·мови + 0,15·музей + 0,15·тип.
   Найбільші ваги в статті у Вікіпедії/Wikidata і в перекладах назви, бо це рішення сторонньої людини:
   хтось сам вирішив описати об'єкт або перекласти назву. Музей і конкретний тип історичного об'єкта
   корисні, але важать менше. Площу не включила, бо величина не обов'язково говорить про цінність для туриста.

3. У топ-10 за моїм рейтингом і за площею збіглося лише 2 об'єкти (Художній музей і музей Коцюбинського). За площею в топ потрапив навіть «Нудистський пляж», а пам'ятники туди потрапити не можуть, бо в точок
   площа нуль.

4. Коефіцієнт обходу 1,21 для малої задачі і 1,17 для великої. Пам'ятки малої задачі стоять у компактному центрі з густою мережею вулиць.
   У великій задачі точки розкидані ширше, відстані довші, і дрібні обходи будинків менше впливають
   на загальну довжину. Асиметрія 0 м, бо пішохід може йти будь-якою вулицею в обидва боки.

5. Найближчий сусід гірший за точний розв'язок на 7,97 %. 2-opt скоротив помилку до 2,91 %, але оптимуму не досяг. На 30 точках перевірити це вже неможливо, тому ми не знаємо, наскільки 33 км далекі від оптимуму.

6. Через пам'ять. Таблиця Held-Karp має 2^(n-1)·(n-1) клітин. Для 31 точки (вокзал і 30 пам'яток) це 32 212 254 720 клітин, тобто близько 258 ГБ при 8 байтах на клітину. Кожна нова точка подвоює таблицю: для 21 точки ще 0,17 ГБ, для 26 вже 6,7 ГБ.

7. Що виявилось неправильним:
   - спершу геокодер повернув не місто, а всю Чернігівську область;
     виправила, вибравши результат з типом city і правильною площею;
   - «Чорна могила» записана двічі (як memorial і як tomb, 7 м одна від одної); залишила memorial;
   - заповідник «Чернігів стародавній» охоплює собори, тож видалила його як окрему зупинку;
   - 33 об'єкти без назви;
   - військова техніка (кожен літак і гармата окремим об'єктом) утворила найгустіші комірки H3, а частина з неї потрапили у топ-30 і подовжили маршрут. На майбутнє краще виключати вручну такі об'єкти, а не сподіватись що рейтинг відсіє.


### Декларація про використання штучного інтелекту

Зазначте, чи користувались ШІ-помічником, у яких саме місцях і як перевіряли
отриманий код. Використання дозволене; **неперевірене** використання — ні.

In [92]:
REPORT = "Декларація про використання ШІ:\nПід час виконання роботи я користувалася ШІ-помічником Claude (Anthropic).\nДе саме:\n- для пояснення суті завдань і коду, який був у ноутбуці.\n- для написання коду.\nРішення ухвалювала сама: які типи об'єктів відкинути, які сигнали і ваги взяти в рейтинг, що робити з дублікатами, який вокзал обрати тощо."
print(REPORT)

Декларація про використання ШІ:
Під час виконання роботи я користувалася ШІ-помічником Claude (Anthropic).
Де саме:
- для пояснення суті завдань і коду, який був у ноутбуці.
- для написання коду.
Рішення ухвалювала сама: які типи об'єктів відкинути, які сигнали і ваги взяти в рейтинг, що робити з дублікатами, який вокзал обрати тощо.
